In [ ]:
# Loading libraries needed for the analysis

# Geolibraries
import geopandas as gpd
import osmnx as ox
import contextily as ctx; import basemaps

# Routing
from r5py import TravelTimeMatrixComputer, TransportMode
import datetime
from datetime import timedelta

# R5
import r5py
from r5py import TransportNetwork

# General tools
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from h3 import h3
from shapely.geometry import Polygon
from shapely import wkt

In [ ]:
## Network Santiago
# Saving the path for .pbf and gtfs archives into new variables
osm_stgo = "./data/Helsinki_larger_region.osm.pbf"
gtfs_stgo = "./data/GTFS_helsinki.zip"

# Build de network
network = TransportNetwork(osm_stgo,[gtfs_stgo])

In [ ]:
network

In [ ]:
## Oringins and destinations (Points)
stgo_hexagons_9_origins = stgo_hexagons_9_origins.set_geometry("centroids")
stgo_hexagons_9_destinations = stgo_hexagons_9_destinations.set_geometry("centroids")

In [ ]:
hsk_shape = gpd.read_file("./data/h3_polygons_Helsinki.gpkg")

In [ ]:
from shapely.geometry import Point

# Assuming hsk_shape is a GeoDataFrame with hexagon geometries
hsk_centroids = hsk_shape.copy()
hsk_centroids['geometry'] = hsk_centroids.centroid

# Optional: ensure the CRS is consistent
hsk_centroids.set_crs(hsk_shape.crs, inplace=True)

In [ ]:
# Rename column 'ID' to 'id'
hsk_centroids = hsk_centroids.rename(columns={"ID": "id"})

In [ ]:
# Creating two df for origins and destinations
hsk_hexagons_9_origins = hsk_centroids.copy()
hsk_hexagons_9_destinations = hsk_centroids.copy()

In [ ]:
## Travel time matrix
travel_time_matrix_computer = TravelTimeMatrixComputer(
    network,
    origins=hsk_hexagons_9_origins,
    destinations=hsk_hexagons_9_destinations,
    departure=datetime.datetime(2024,3,23,8,0), # Thursday
    max_time = timedelta(minutes=45),
    departure_time_window = timedelta(minutes=5), # Using a window of 30 min 
    transport_modes=[TransportMode.BICYCLE],
    
)
travel_time_matrix = travel_time_matrix_computer.compute_travel_times()

In [ ]:
travel_time_matrix_computer

In [ ]:
# Saving the travel time matrix
travel_time_matrix.to_csv("./data/travel_time_matrix_bicycle.csv")

In [ ]:
# Saving the travel time matrix
travel_time_matrix = pd.read_csv("./data/travel_time_matrix_bicycle.csv")

In [ ]:
travel_time_matrix.head(30)

In [ ]:
travel_time_matrix.isna().sum()

In [ ]:
# travel_time_matrix is the DataFrame to update
travel_time_matrix = travel_time_matrix.dropna()

# Reset the index afterwards
travel_time_matrix.reset_index(drop=True, inplace=True)

In [ ]:
from shapely.geometry import Polygon
import h3

# Get unique from_id values
unique_from_ids = travel_time_matrix['from_id'].unique()

# Convert H3 indices to polygons
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Create GeoDataFrame
geometry = [h3_to_polygon(h) for h in unique_from_ids]
gdf_from_hexes = gpd.GeoDataFrame({'from_id': unique_from_ids}, geometry=geometry, crs='EPSG:4326')

# Preview result
gdf_from_hexes.head()

In [ ]:
gdf_from_hexes.explore()

In [ ]:
# First, create a proper copy to avoid ambiguity
travel_time_matrix = travel_time_matrix.copy()

# Calculate distance in kilometers (assuming average speed is 12 km/h => 0.2 km/min)
travel_time_matrix.loc[:, 'distance_km'] = travel_time_matrix['travel_time'] * 0.2

# Calculate CO2 emissions in grams (21g CO2 per km)
travel_time_matrix.loc[:, 'bike_co2'] = travel_time_matrix['distance_km'] * 21

In [ ]:
df_bike_co2 = travel_time_matrix.copy()

In [ ]:
df_bike_co2.to_parquet("./data/co2_travel_time_bike.parquet")

In [ ]:
hsk_pois = pd.read_parquet("./data/pois_per_hex.parquet")

In [ ]:
# Filter rows where pt_time is less than or equal to 15, 30, and 45
df_bike_co2_250 = df_bike_co2[df_bike_co2["bike_co2"] <= 250].copy()
df_bike_co2_500 = df_bike_co2[df_bike_co2["bike_co2"] <= 500].copy()
df_bike_co2_125 = df_bike_co2[df_bike_co2["bike_co2"] <= 125].copy()

In [ ]:
# Step 1: Group hsk_pois by 'h3_id' and 'category' to get counts
pois_grouped = hsk_pois.groupby(['h3_id', 'category']).size().unstack(fill_value=0).reset_index()

# Step 2: Merge with df_pt_15 using 'to_id' (in df_pt_15) and 'h3_id' (in pois_grouped)
df_bike_co2_250_merged = df_bike_co2_250.merge(pois_grouped, how='left', left_on='to_id', right_on='h3_id')
df_bike_co2_250_merged = df_bike_co2_250_merged.drop(columns=['h3_id'])

# Step 3: Merge again using 'from_id' to get POIs at origin
from_pois = df_bike_co2_250.merge(pois_grouped, how='left', left_on='from_id', right_on='h3_id')
from_pois = from_pois.drop(columns=['h3_id'])

# Step 4: Add category columns from origin and destination
category_cols = pois_grouped.columns.drop('h3_id')
for col in category_cols:
    df_bike_co2_250_merged[col] = df_bike_co2_250_merged[col].fillna(0) + from_pois[col].fillna(0)

In [ ]:
# Step 5: Group by 'from_id', sum POI categories, and average pt_co2
category_cols = [
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Uncategorized',
    'Well-being & Lifestyle'
]

grouped_summary = df_bike_co2_250_merged.groupby('from_id')[category_cols + ['travel_time']].agg({
    'Educational Facilities': 'sum',
    'Grocery Stores & Supermarkets': 'sum',
    'Jobs, Professional Services & Religious': 'sum',
    'Restaurant & Entertainment': 'sum',
    'Shopping & Retail': 'sum',
    'Uncategorized': 'sum',
    'Well-being & Lifestyle': 'sum',
    'travel_time':'mean'
    
}).reset_index()

In [ ]:
# Step 6: Create total_pois column (excluding 'Uncategorized')
grouped_summary['total_pois'] = grouped_summary[[
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]].sum(axis=1)

In [ ]:
import geopandas as gpd
import h3
from shapely.geometry import Polygon
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt
import mapclassify
import matplotlib.patches as mpatches


# Function to convert h3 to polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Convert 'from_id' to geometry
grouped_summary = grouped_summary.copy()
geometry = grouped_summary['from_id'].apply(h3_to_polygon)
gdf_hexes = gpd.GeoDataFrame(grouped_summary, geometry=geometry, crs='EPSG:4326')

# Classify total_pois into 5 natural breaks
classifier = mapclassify.NaturalBreaks(y=gdf_hexes['total_pois'], k=5)
gdf_hexes['poi_class'] = classifier.yb

# Get bin edges for legend
bin_edges = classifier.bins

# Define custom legend handles
legend_handles = []
for i in range(len(bin_edges)):
    if i == 0:
        label = f"<= {bin_edges[i]:.1f}"
    else:
        label = f"> {bin_edges[i-1]:.1f} – {bin_edges[i]:.1f}"
    patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges)-1)), label=label)
    legend_handles.append(patch)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
ax.set_title("Total POIs per Hexagon (Natural Breaks)")
ax.axis('off')
plt.legend(handles=legend_handles, title="Total POIs", loc='lower left')
plt.tight_layout()
plt.show()

In [ ]:
# Function to convert h3 index to shapely Polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Define POI categories to plot
poi_categories = [
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]

# Create GeoDataFrame with geometry if not already created
if 'geometry' not in grouped_summary.columns:
    geometry = grouped_summary['from_id'].apply(h3_to_polygon)
    gdf_hexes = gpd.GeoDataFrame(grouped_summary, geometry=geometry, crs='EPSG:4326')
else:
    gdf_hexes = grouped_summary.copy()

# Set up plot grid
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

# Generate map for each category
for idx, category in enumerate(poi_categories):
    ax = axes[idx]
    # Classify with Natural Breaks
    classifier = mapclassify.NaturalBreaks(y=gdf_hexes[category], k=5)
    gdf_hexes['poi_class'] = classifier.yb
    bin_edges = classifier.bins

    # Create custom legend
    legend_handles = []
    for i in range(len(bin_edges)):
        if i == 0:
            label = f"<= {bin_edges[i]:.1f}"
        else:
            label = f"> {bin_edges[i-1]:.1f} – {bin_edges[i]:.1f}"
        patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges) - 1)), label=label)
        legend_handles.append(patch)

    # Plot
    gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
    ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
    ax.set_title(category)
    ax.axis('off')
    ax.legend(handles=legend_handles, title=category, loc='lower left')

plt.suptitle("POI Categories by Hexagon in 250g trip by bike", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [ ]:
# Filter rows where pt_time is less than or equal to 15, 30, and 45
df_bike_15 = df_bike_co2[df_bike_co2["travel_time"] <= 15].copy()
df_bike_30 = df_bike_co2[df_bike_co2["travel_time"] <= 30].copy()
df_bike_45 = df_bike_co2[df_bike_co2["travel_time"] <= 45].copy()

In [ ]:
# Step 1: Group hsk_pois by 'h3_id' and 'category' to get counts
pois_grouped = hsk_pois.groupby(['h3_id', 'category']).size().unstack(fill_value=0).reset_index()

# Step 2: Merge with df_pt_15 using 'to_id' (in df_pt_15) and 'h3_id' (in pois_grouped)
df_bike_30_merged = df_bike_30.merge(pois_grouped, how='left', left_on='to_id', right_on='h3_id')
df_bike_30_merged = df_bike_30_merged.drop(columns=['h3_id'])

# Step 3: Merge again using 'from_id' to get POIs at origin
from_pois = df_bike_30.merge(pois_grouped, how='left', left_on='from_id', right_on='h3_id')
from_pois = from_pois.drop(columns=['h3_id'])

# Step 4: Add category columns from origin and destination
category_cols = pois_grouped.columns.drop('h3_id')
for col in category_cols:
    df_bike_30_merged[col] = df_bike_30_merged[col].fillna(0) + from_pois[col].fillna(0)

In [ ]:
category_cols = [
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Uncategorized',
    'Well-being & Lifestyle'
]

grouped_summary_30 = df_bike_30_merged.groupby('from_id')[category_cols + ['travel_time']].agg({
    'Educational Facilities': 'sum',
    'Grocery Stores & Supermarkets': 'sum',
    'Jobs, Professional Services & Religious': 'sum',
    'Restaurant & Entertainment': 'sum',
    'Shopping & Retail': 'sum',
    'Uncategorized': 'sum',
    'Well-being & Lifestyle': 'sum',
    'travel_time':'mean'
    
}).reset_index()


In [ ]:
# Step 6: Create total_pois column (excluding 'Uncategorized')
grouped_summary_30['total_pois'] = grouped_summary_30[[
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]].sum(axis=1)

In [ ]:


# Function to convert h3 to polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Convert 'from_id' to geometry
grouped_summary_30 = grouped_summary_30.copy()
geometry = grouped_summary_30['from_id'].apply(h3_to_polygon)
gdf_hexes = gpd.GeoDataFrame(grouped_summary_30, geometry=geometry, crs='EPSG:4326')

# Classify total_pois into 5 natural breaks
classifier = mapclassify.NaturalBreaks(y=gdf_hexes['total_pois'], k=5)
gdf_hexes['poi_class'] = classifier.yb

# Get bin edges for legend
bin_edges = classifier.bins

# Define custom legend handles
legend_handles = []
for i in range(len(bin_edges)):
    if i == 0:
        label = f"<= {bin_edges[i]:.1f}"
    else:
        label = f"> {bin_edges[i-1]:.1f} – {bin_edges[i]:.1f}"
    patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges)-1)), label=label)
    legend_handles.append(patch)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
ax.set_title("Total POIs per Hexagon (Natural Breaks) 30min")
ax.axis('off')
plt.legend(handles=legend_handles, title="Total POIs", loc='lower left')
plt.tight_layout()
plt.show()

In [ ]:

def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Define POI categories to plot
poi_categories = [
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]

# Create GeoDataFrame with geometry if not already created
if 'geometry' not in grouped_summary_30.columns:
    geometry = grouped_summary_30['from_id'].apply(h3_to_polygon)
    gdf_hexes = gpd.GeoDataFrame(grouped_summary_30, geometry=geometry, crs='EPSG:4326')
else:
    gdf_hexes = grouped_summary_30.copy()


# Set up plot grid
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

# Generate map for each category
for idx, category in enumerate(poi_categories):
    ax = axes[idx]
    # Classify with Natural Breaks
    classifier = mapclassify.NaturalBreaks(y=gdf_hexes[category], k=5)
    gdf_hexes['poi_class'] = classifier.yb
    bin_edges = classifier.bins

    # Create custom legend
    legend_handles = []
    for i in range(len(bin_edges)):
        if i == 0:
            label = f"<= {bin_edges[i]:.1f}"
        else:
            label = f"> {bin_edges[i-1]:.1f} – {bin_edges[i]:.1f}"
        patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges) - 1)), label=label)
        legend_handles.append(patch)

    # Plot
    gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
    ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
    ax.set_title(category)
    ax.axis('off')
    ax.legend(handles=legend_handles, title=category, loc='lower left')

plt.suptitle("POI Categories by Hexagon in 30min trip (Natural Breaks)", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# Step 1: Group hsk_pois by 'h3_id' and 'category' to get counts
pois_grouped = hsk_pois.groupby(['h3_id', 'category']).size().unstack(fill_value=0).reset_index()

# Step 2: Merge with df_bike_co2_125 using 'to_id' (in df_bike_co2_125) and 'h3_id' (in pois_grouped)
df_bike_co2_125_merged = df_bike_co2_125.merge(pois_grouped, how='left', left_on='to_id', right_on='h3_id')
df_bike_co2_125_merged = df_bike_co2_125_merged.drop(columns=['h3_id'])

# Step 3: Merge again using 'from_id' to get POIs at origin
from_pois = df_bike_co2_125.merge(pois_grouped, how='left', left_on='from_id', right_on='h3_id')
from_pois = from_pois.drop(columns=['h3_id'])

# Step 4: Add category columns from origin and destination
category_cols = pois_grouped.columns.drop('h3_id')
for col in category_cols:
    df_bike_co2_125_merged[col] = df_bike_co2_125_merged[col].fillna(0) + from_pois[col].fillna(0)

# Step 5: Group by 'from_id', sum POI categories, and average travel_time
grouped_summary_125 = df_bike_co2_125_merged.groupby('from_id')[category_cols.tolist() + ['travel_time']].agg({
    'Educational Facilities': 'sum',
    'Grocery Stores & Supermarkets': 'sum',
    'Jobs, Professional Services & Religious': 'sum',
    'Restaurant & Entertainment': 'sum',
    'Shopping & Retail': 'sum',
    'Uncategorized': 'sum',
    'Well-being & Lifestyle': 'sum',
    'travel_time': 'mean'
}).reset_index()

# Step 6: Create total_pois column (excluding 'Uncategorized')
grouped_summary_125['total_pois'] = grouped_summary_125[[
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]].sum(axis=1)

# Convert 'from_id' to geometry
import geopandas as gpd
import h3
from shapely.geometry import Polygon
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt
import mapclassify
import matplotlib.patches as mpatches

def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

grouped_summary_125 = grouped_summary_125.copy()
geometry = grouped_summary_125['from_id'].apply(h3_to_polygon)
gdf_hexes_125 = gpd.GeoDataFrame(grouped_summary_125, geometry=geometry, crs='EPSG:4326')

# Classify total_pois into 5 natural breaks
classifier = mapclassify.NaturalBreaks(y=gdf_hexes_125['total_pois'], k=5)
gdf_hexes_125['poi_class'] = classifier.yb
bin_edges = classifier.bins

# Define custom legend handles
legend_handles = []
for i in range(len(bin_edges)):
    if i == 0:
        label = f"<= {bin_edges[i]:.1f}"
    else:
        label = f"> {bin_edges[i-1]:.1f} – {bin_edges[i]:.1f}"
    patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges)-1)), label=label)
    legend_handles.append(patch)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
gdf_hexes_125.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes_125.crs.to_string())
ax.set_title("Total POIs per Hexagon (Natural Breaks) - 125g")
ax.axis('off')
plt.legend(handles=legend_handles, title="Total POIs", loc='lower left')
plt.tight_layout()
plt.show()


In [ ]:

# Function to convert h3 index to shapely polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Define POI categories to plot
poi_categories = [
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]

# Create GeoDataFrame with geometry if not already created
if 'geometry' not in grouped_summary_125.columns:
    geometry = grouped_summary_125['from_id'].apply(h3_to_polygon)
    gdf_hexes = gpd.GeoDataFrame(grouped_summary_125, geometry=geometry, crs='EPSG:4326')
else:
    gdf_hexes = grouped_summary_125.copy()

# Set up plot grid
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

# Generate map for each category
for idx, category in enumerate(poi_categories):
    ax = axes[idx]
    # Classify with Natural Breaks
    classifier = mapclassify.NaturalBreaks(y=gdf_hexes[category], k=5)
    gdf_hexes['poi_class'] = classifier.yb
    bin_edges = classifier.bins

    # Create custom legend
    legend_handles = []
    for i in range(len(bin_edges)):
        if i == 0:
            label = f"<= {bin_edges[i]:.1f}"
        else:
            label = f"> {bin_edges[i-1]:.1f} – {bin_edges[i]:.1f}"
        patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges) - 1)), label=label)
        legend_handles.append(patch)

    # Plot
    gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
    ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
    ax.set_title(category)
    ax.axis('off')
    ax.legend(handles=legend_handles, title=category, loc='lower left')

plt.suptitle("Reachable POIs by Hexagon in a 125 CO2g trip", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()
